# NiyamTrace-X — Paper Closure Notebook

**Purpose:** close the remaining external-validation gaps and emit a final evidence package that can be integrated into the manuscript without hand-reconciling benchmark outputs.

This notebook does **not** rerun the already-frozen NiyamTrace 2k, ablation, Anchor Lock V3, or VEB V2 experiments.

## Closure target

The notebook tries to obtain **native numeric evidence** from all three required external benchmark families:

1. **BFCL-v4** — function calling.
2. **AgentDojo** — prompt-injection utility/security.
3. **τ³ / tau2-bench** — stateful customer-service tool use.

It searches for **at least three independent LLM families** that pass a live tool-calling preflight. Qwen, GPT-OSS, GLM, Llama, and Gemini are candidate families; the first three viable families are used, with automatic fallbacks inside each family.

## Important evidence rule

A benchmark is never marked complete merely because a command exits with code 0.

`SUPPORTED` requires official/native result files, evaluated cases, numeric benchmark-native scores, and no infrastructure error for the claimed slice.

## Output

- `NTX_CLOSURE_RAW_DATA.zip`
- `NTX_CLOSURE_PROCESSED_RESULTS.zip`
- `NTX_CLOSURE_PAPER_INTEGRATION.zip`
- `NTX_PAPER_CLOSURE_MASTER.zip`

The master ZIP is the file to send back for final PDF regeneration.

In [ ]:
# CELL 1 — CONFIGURATION
from pathlib import Path
from getpass import getpass
from datetime import datetime, timezone
import os, sys, json, re, time, random, hashlib, zipfile, shutil, subprocess, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED)
np.random.seed(SEED)

SELFTEST=os.getenv("NTX_SELFTEST","0")=="1"
MODE=os.getenv("NTX_CLOSURE_MODE","CLOSURE").upper()
assert MODE in {"SMOKE","CLOSURE","FULL"}

BASE=Path(os.getenv(
    "NTX_CLOSURE_DIR",
    "/content/NTX_PAPER_CLOSURE" if Path("/content").exists() else str(Path.cwd()/"NTX_PAPER_CLOSURE")
))
WORK=BASE/"work"; RAW=BASE/"raw"; RESULTS=BASE/"results"; LOGS=BASE/"logs"
ENV=BASE/"environment"; PAPER=BASE/"paper_integration"; ARCH=BASE/"archives"
for p in [BASE,WORK,RAW,RESULTS,LOGS,ENV,PAPER,ARCH]:
    p.mkdir(parents=True,exist_ok=True)

CFG={
    "SMOKE":{
        "families_needed":3,"bfcl_limit":5,
        "dojo_suites":["banking"],"dojo_user_tasks":["user_task_0"],
        "dojo_injection_tasks":["injection_task_0"],
        "tau_domains":["airline","retail","telecom"],"tau_tasks_per_domain":1,
        "tau_max_steps":18,"tau_agent_max_tokens":768,"tau_user_max_tokens":384,
        "bootstrap":500,
    },
    "CLOSURE":{
        "families_needed":3,"bfcl_limit":100,
        "dojo_suites":["banking","workspace","travel","slack"],
        "dojo_user_tasks":["user_task_0","user_task_5","user_task_10"],
        "dojo_injection_tasks":["injection_task_0","injection_task_1"],
        "tau_domains":["airline","retail","telecom"],"tau_tasks_per_domain":10,
        "tau_max_steps":50,"tau_agent_max_tokens":1536,"tau_user_max_tokens":768,
        "bootstrap":5000,
    },
    "FULL":{
        "families_needed":3,"bfcl_limit":None,
        "dojo_suites":["banking","workspace","travel","slack"],
        "dojo_user_tasks":None,"dojo_injection_tasks":None,
        "tau_domains":["airline","retail","telecom"],"tau_tasks_per_domain":None,
        "tau_max_steps":100,"tau_agent_max_tokens":2048,"tau_user_max_tokens":1024,
        "bootstrap":10000,
    }
}[MODE]

MAX_PRICE_PROXY=float(os.getenv("NTX_MAX_PRICE_PROXY","inf"))
ENABLE_AGENTDYN=os.getenv("NTX_ENABLE_AGENTDYN","0")=="1"
ENABLE_MCP=os.getenv("NTX_ENABLE_MCP","0")=="1"
MLCL_OFFICIAL_PATH=os.getenv("MLCL_OFFICIAL_PATH","").strip()
PREVIOUS_RESULTS_ZIP=os.getenv("NTX_PREVIOUS_RESULTS_ZIP","").strip()

print("SELFTEST:",SELFTEST)
print("MODE:",MODE)
print(json.dumps(CFG,indent=2))

In [ ]:
# CELL 2 — PROCESS / CHECKPOINT / ARCHIVE HELPERS
CHECKPOINT=RESULTS/"closure_checkpoint.json"

def now():
    return datetime.now(timezone.utc).isoformat()

def run(cmd,cwd=None,env=None,timeout=None):
    return subprocess.run(
        [str(x) for x in cmd],cwd=str(cwd) if cwd else None,env=env,
        capture_output=True,text=True,errors="replace",timeout=timeout
    )

def save_log(name,p,cmd=None):
    parts=[]
    if cmd is not None: parts += ["COMMAND"," ".join(map(str,cmd)),""]
    parts += ["STDOUT",p.stdout or "","STDERR",p.stderr or ""]
    (LOGS/name).write_text("\n".join(parts),errors="ignore")

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for c in iter(lambda:f.read(1024*1024),b""): h.update(c)
    return h.hexdigest()

def save_json(path,obj):
    Path(path).write_text(json.dumps(obj,indent=2,default=str))

def load_json(path,default=None):
    try:return json.loads(Path(path).read_text())
    except Exception:return {} if default is None else default

STATE=load_json(CHECKPOINT,{})
def ck_get(key): return STATE.get(key,{})
def ck_set(key,status,**extra):
    STATE[key]={"status":status,"updated_at":now(),**extra}
    save_json(CHECKPOINT,STATE)
def supported(key): return ck_get(key).get("status")=="SUPPORTED"

def ensure_uv():
    if not shutil.which("uv"):
        p=run([sys.executable,"-m","pip","install","-q","-U","uv"])
        save_log("install_uv.log",p)
        if p.returncode: raise RuntimeError("uv installation failed")

def clone(url,dest):
    dest=Path(dest)
    if not dest.exists():
        p=run(["git","clone","--depth","1",url,dest])
        save_log(f"clone_{dest.name}.log",p)
        if p.returncode: raise RuntimeError(f"clone failed: {url}")
    commit=run(["git","-C",dest,"rev-parse","HEAD"]).stdout.strip()
    status=run(["git","-C",dest,"status","--porcelain"]).stdout
    return commit,status

def copytree(src,dst):
    src,dst=Path(src),Path(dst)
    if not src.exists(): return False
    shutil.rmtree(dst,ignore_errors=True)
    shutil.copytree(src,dst)
    return True

def make_zip(src,dst):
    src,dst=Path(src),Path(dst)
    if dst.exists():dst.unlink()
    with zipfile.ZipFile(dst,"w",zipfile.ZIP_DEFLATED,allowZip64=True) as z:
        for p in sorted(src.rglob("*")):
            if p.is_file(): z.write(p,arcname=str(p.relative_to(src)))
    return dst

def classify_error(text):
    t=(text or "").lower()
    if any(x in t for x in ["402","payment required","credits","budget","can only afford"]): return "PROVIDER_BUDGET"
    if any(x in t for x in ["429","rate limit","too many requests"]): return "RATE_LIMIT"
    if any(x in t for x in ["context length","maximum context","too many tokens"]): return "CONTEXT_LIMIT"
    if any(x in t for x in ["not found","404","invalid model","model does not exist"]): return "MODEL_UNAVAILABLE"
    if any(x in t for x in ["module not found","modulenotfounderror","importerror"]): return "DEPENDENCY"
    return "OTHER"

print("Checkpoint entries:",len(STATE))

## Model-family recovery ladder

The notebook queries the live OpenRouter catalog and tries preferred candidates first. If one candidate fails, it searches for another model from the same independent family. A candidate must pass both ordinary chat and native tool calling before entering the benchmark matrix.

In [ ]:
# CELL 3 — OPENROUTER CATALOG / FAMILY DISCOVERY
import urllib.request, urllib.error
API_BASE="https://openrouter.ai/api/v1"

if SELFTEST:
    OPENROUTER_API_KEY="SELFTEST"
else:
    OPENROUTER_API_KEY=os.getenv("OPENROUTER_API_KEY","").strip()
    if not OPENROUTER_API_KEY:
        OPENROUTER_API_KEY=getpass("OpenRouter API key (hidden): ").strip()
    if not OPENROUTER_API_KEY: raise RuntimeError("OPENROUTER_API_KEY required")

def http_json(method,url,body=None,timeout=60):
    headers={"Authorization":"Bearer "+OPENROUTER_API_KEY,"Content-Type":"application/json",
             "HTTP-Referer":"https://openai.com/","X-Title":"NiyamTrace-X Paper Closure"}
    data=None if body is None else json.dumps(body).encode()
    req=urllib.request.Request(url,data=data,headers=headers,method=method)
    with urllib.request.urlopen(req,timeout=timeout) as r:
        return r.status,json.loads(r.read().decode())

if SELFTEST:
    catalog={"data":[
        {"id":"qwen/qwen3-30b-a3b-instruct-2507","context_length":131072,"pricing":{"prompt":"0.00000005","completion":"0.0000002"}},
        {"id":"openai/gpt-oss-20b","context_length":131072,"pricing":{"prompt":"0.0000001","completion":"0.0000005"}},
        {"id":"z-ai/glm-4.5-air","context_length":131072,"pricing":{"prompt":"0.00000013","completion":"0.00000085"}},
        {"id":"meta-llama/llama-3.3-70b-instruct","context_length":131072,"pricing":{"prompt":"0.00000012","completion":"0.0000003"}},
        {"id":"google/gemini-2.5-flash","context_length":1000000,"pricing":{"prompt":"0.00000015","completion":"0.0000006"}},
    ]}
    key_status={"selftest":True}
else:
    _,catalog=http_json("GET",API_BASE+"/models")
    try: _,key_status=http_json("GET",API_BASE+"/key")
    except Exception as e: key_status={"status":"UNAVAILABLE","error":repr(e)}

save_json(ENV/"openrouter_catalog.json",catalog)
save_json(ENV/"openrouter_key_status_redacted.json",{
    k:v for k,v in key_status.items()
    if not any(x in k.lower() for x in ["token","key","secret"])
})

PREFERRED={
    "Qwen":["qwen/qwen3-30b-a3b-instruct-2507","qwen/qwen3-coder:exacto","qwen/qwen3-235b-a22b-2507"],
    "GPT-OSS":["openai/gpt-oss-20b","openai/gpt-oss-120b:exacto","openai/gpt-oss-120b"],
    "GLM":["z-ai/glm-4.5-air","z-ai/glm-4.6:exacto","z-ai/glm-4.6"],
    "Llama":["meta-llama/llama-3.3-70b-instruct"],
    "Gemini":["google/gemini-2.5-flash"],
}
FAMILY_MATCH={
    "Qwen":lambda x:x.startswith("qwen/"),
    "GPT-OSS":lambda x:"gpt-oss" in x,
    "GLM":lambda x:x.startswith("z-ai/glm"),
    "Llama":lambda x:x.startswith("meta-llama/"),
    "Gemini":lambda x:x.startswith("google/gemini"),
}
rows=[x for x in catalog.get("data",[]) if isinstance(x,dict) and x.get("id")]
byid={x["id"]:x for x in rows}

def price_proxy(row):
    try:return float((row.get("pricing") or {}).get("prompt","inf"))+float((row.get("pricing") or {}).get("completion","inf"))
    except Exception:return float("inf")

def context_len(row):
    try:return int(row.get("context_length") or 0)
    except Exception:return 0

family_candidates={}
for family,matcher in FAMILY_MATCH.items():
    ordered=[]
    for mid in PREFERRED.get(family,[]):
        if mid in byid and mid not in ordered: ordered.append(mid)
    dynamic=[r for r in rows if matcher(r["id"]) and context_len(r)>=32768 and price_proxy(r)<=MAX_PRICE_PROXY]
    dynamic=sorted(dynamic,key=lambda r:(price_proxy(r),-context_len(r)))
    for r in dynamic:
        if r["id"] not in ordered: ordered.append(r["id"])
    family_candidates[family]=ordered[:8]

save_json(ENV/"family_candidates.json",family_candidates)
print(json.dumps(family_candidates,indent=2))

In [ ]:
# CELL 4 — CHAT + TOOL-CALL PREFLIGHT; PICK THREE FAMILIES
tool=[{"type":"function","function":{
    "name":"lookup_order","description":"Lookup an order",
    "parameters":{"type":"object","properties":{"order_id":{"type":"string"}},"required":["order_id"]}
}}]

def chat(model,messages,tools=None,max_tokens=96):
    body={"model":model,"messages":messages,"temperature":0,"max_tokens":max_tokens}
    if tools is not None:
        body["tools"]=tools; body["tool_choice"]="auto"
    _,o=http_json("POST",API_BASE+"/chat/completions",body)
    return o

preflight=[]; selected=[]

if SELFTEST:
    for family in ["Qwen","GPT-OSS","GLM"]:
        model=family_candidates[family][0]
        preflight.append({"family":family,"model":model,"chat_ok":True,"tool_ok":True,"status":"SELFTEST_ONLY"})
        selected.append({"family":family,"model":model,"label":family.lower().replace("-","_")})
else:
    for family,cands in family_candidates.items():
        chosen=None
        for model in cands:
            row={"family":family,"model":model}
            try:
                a=chat(model,[{"role":"user","content":"Reply exactly OK"}],max_tokens=16)
                row["chat_ok"]=bool(a.get("choices"))
            except Exception as e:
                row["chat_ok"]=False; row["chat_error"]=repr(e)
            try:
                b=chat(model,[{"role":"user","content":"Use lookup_order for order A123."}],tools=tool,max_tokens=96)
                msg=(b.get("choices") or [{}])[0].get("message") or {}
                row["tool_ok"]=bool(msg.get("tool_calls"))
            except Exception as e:
                row["tool_ok"]=False; row["tool_error"]=repr(e)
            row["status"]="OK" if row.get("chat_ok") and row.get("tool_ok") else "FAILED"
            preflight.append(row)
            if row["status"]=="OK":
                chosen=model; break
        if chosen:
            selected.append({"family":family,"model":chosen,"label":family.lower().replace("-","_")})
        if len(selected)>=CFG["families_needed"]: break

preflight_df=pd.DataFrame(preflight)
preflight_df.to_csv(RESULTS/"00_model_preflight.csv",index=False)
save_json(ENV/"selected_model_families.json",selected)
display(preflight_df)
print("Selected:",selected)

if not SELFTEST and len(selected)<CFG["families_needed"]:
    raise RuntimeError(f"Only {len(selected)} independent model families passed chat+tool preflight; {CFG['families_needed']} required.")

In [ ]:
# CELL 5 — IMPORT PREVIOUS MERGED RESULTS IF AVAILABLE
previous_import={"status":"NOT_PROVIDED"}
candidate_paths=[]
if PREVIOUS_RESULTS_ZIP: candidate_paths.append(Path(PREVIOUS_RESULTS_ZIP))
candidate_paths += [Path("/content/NTX_RESULTS_MERGED_WITH_RECOVERY.zip"),Path.cwd()/"NTX_RESULTS_MERGED_WITH_RECOVERY.zip"]
prev=next((p for p in candidate_paths if p.exists()),None)
if prev:
    prev_dir=RAW/"previous_results"
    shutil.rmtree(prev_dir,ignore_errors=True); prev_dir.mkdir(parents=True)
    with zipfile.ZipFile(prev) as z: z.extractall(prev_dir)
    previous_import={"status":"IMPORTED","source":str(prev),"sha256":sha256_file(prev),
                     "files":sum(1 for p in prev_dir.rglob("*") if p.is_file())}
save_json(RESULTS/"01_previous_results_import.json",previous_import)
print(previous_import)

# Stage A — BFCL-v4

Uses a clean Python 3.11 EvalScope environment and the supported `evalscope[bfcl]` dependency bundle. Provider/model failures trigger fallback within the same model family.

In [ ]:
# CELL 6 — BFCL ENVIRONMENT
BFENV=WORK/"bfcl_env"
if SELFTEST:
    BFPY=Path(sys.executable)
else:
    ensure_uv()
    run(["uv","python","install","3.11"])
    if not BFENV.exists():
        p=run(["uv","venv",BFENV,"--python","3.11"]); save_log("bfcl_venv.log",p)
        if p.returncode:raise RuntimeError("BFCL venv creation failed")
    BFPY=BFENV/"bin"/"python"
    p=run(["uv","pip","install","--python",BFPY,"-U","evalscope[bfcl]"]); save_log("bfcl_install.log",p)
    if p.returncode:raise RuntimeError("BFCL dependency install failed")
    v=run([BFPY,"-c","from evalscope import run_task; from evalscope.config import TaskConfig; print('BFCL_READY')"])
    save_log("bfcl_verify.log",v)
    if v.returncode or "BFCL_READY" not in v.stdout:raise RuntimeError("BFCL verification failed")
    fr=run([BFPY,"-m","pip","freeze"]); (ENV/"bfcl_pip_freeze.txt").write_text(fr.stdout or "")
print("BFCL Python:",BFPY)

In [ ]:
# CELL 7 — BFCL EXECUTION / FALLBACK / PARSING
def parse_bfcl(root,label,family):
    root=Path(root); scores=[]; evaluated=0
    for p in root.rglob("*"):
        if not p.is_file():continue
        rel=str(p.relative_to(root)); low=rel.lower()
        try:
            if p.suffix.lower()==".csv":
                df=pd.read_csv(p)
                if any(x in low for x in ["report","review","prediction"]): evaluated=max(evaluated,len(df))
                for c in df.columns:
                    if any(x in str(c).lower() for x in ["accuracy","score"]):
                        for v in pd.to_numeric(df[c],errors="coerce").dropna():
                            scores.append({"benchmark":"BFCL-v4","model":label,"family":family,"slice":rel,
                                           "metric":str(c),"score":float(v),"source_file":rel})
            elif p.suffix.lower() in {".json",".jsonl"}:
                objs=[]
                if p.suffix.lower()==".jsonl":
                    for line in p.read_text(errors="ignore").splitlines():
                        try:objs.append(json.loads(line))
                        except Exception:pass
                else:
                    try:objs=[json.loads(p.read_text(errors="ignore"))]
                    except Exception:objs=[]
                stack=[("",o) for o in objs]
                while stack:
                    path,o=stack.pop()
                    if isinstance(o,dict):
                        for ck in ["total_count","evaluated_count","num_samples","n_samples"]:
                            if isinstance(o.get(ck),(int,float)) and o[ck]>0:evaluated=max(evaluated,int(o[ck]))
                        for k,v in o.items():
                            q=f"{path}.{k}" if path else k
                            if isinstance(v,(dict,list)):stack.append((q,v))
                            elif isinstance(v,(int,float)) and any(x in k.lower() for x in ["accuracy","score"]):
                                scores.append({"benchmark":"BFCL-v4","model":label,"family":family,"slice":rel,
                                               "metric":q,"score":float(v),"source_file":rel})
                    elif isinstance(o,list):
                        if any(x in low for x in ["review","prediction"]) and o:evaluated=max(evaluated,len(o))
                        for i,v in enumerate(o):stack.append((f"{path}[{i}]",v))
        except Exception:pass
    return (pd.DataFrame(scores).drop_duplicates() if scores else pd.DataFrame()),evaluated

bfcl_parts=[]; bfcl_status=[]

if SELFTEST:
    for s in selected:
        root=RAW/"bfcl"/s["family"]/"selftest"; root.mkdir(parents=True,exist_ok=True)
        (root/"report.json").write_text(json.dumps({"accuracy":0.8,"total_count":5}))
        d,n=parse_bfcl(root,s["label"],s["family"]); d["evidence_state"]="SELFTEST_ONLY"
        bfcl_parts.append(d); bfcl_status.append({"family":s["family"],"model_id":s["model"],"status":"SELFTEST_ONLY","evaluated":n})
else:
    for s in selected:
        family=s["family"]; key=f"bfcl::{family}::{MODE}"
        if supported(key):
            root=RAW/"bfcl"/family/MODE.lower()
            d,n=parse_bfcl(root,s["label"],family)
            if n>0 and len(d):
                bfcl_parts.append(d)
                bfcl_status.append({"family":family,"model_id":ck_get(key).get("model_id"),"status":"SUPPORTED_REUSED","evaluated":n})
                continue
        candidate_order=[]
        for m in [s["model"]]+family_candidates.get(family,[]):
            if m not in candidate_order:candidate_order.append(m)
        success=False; attempts=[]
        for attempt_idx,model_id in enumerate(candidate_order[:5],1):
            root=RAW/"bfcl"/family/MODE.lower(); root.mkdir(parents=True,exist_ok=True)
            script=root/f"run_{attempt_idx}.py"
            script.write_text(
f'''from evalscope import run_task
from evalscope.config import TaskConfig
cfg=TaskConfig(
 model={model_id!r},
 api_url={API_BASE!r},
 api_key={OPENROUTER_API_KEY!r},
 eval_type="openai_api",
 datasets=["bfcl_v4"],
 work_dir={str(root)!r},
 limit={CFG["bfcl_limit"]!r},
 seed={SEED},
 generation_config={{"temperature":0.0,"max_tokens":1024,"retries":1,"retry_interval":2,"timeout":120}},
 dataset_args={{"bfcl_v4":{{"extra_params":{{"is_fc_model":True}}}}}}
)
run_task(task_cfg=cfg)
''')
            p=run([BFPY,script],cwd=root,timeout=None); save_log(f"bfcl_{family}_{attempt_idx}.log",p,[BFPY,script])
            d,n=parse_bfcl(root,family.lower(),family)
            errclass=classify_error((p.stdout or "")+"\n"+(p.stderr or ""))
            attempts.append({"model_id":model_id,"returncode":p.returncode,"evaluated":n,"numeric_rows":len(d),"error_class":errclass})
            if n>0 and len(d):
                d["model_id"]=model_id; bfcl_parts.append(d)
                bfcl_status.append({"family":family,"model_id":model_id,"status":"SUPPORTED","evaluated":n})
                ck_set(key,"SUPPORTED",model_id=model_id,evaluated=n); success=True; break
            if errclass not in {"PROVIDER_BUDGET","RATE_LIMIT","MODEL_UNAVAILABLE","CONTEXT_LIMIT"}:break
        save_json(RESULTS/f"bfcl_{family}_attempts.json",attempts)
        if not success:
            bfcl_status.append({"family":family,"model_id":None,"status":"INFRA_FAILURE","evaluated":0})
            ck_set(key,"INFRA_FAILURE",attempts=attempts)

bfcl=pd.concat(bfcl_parts,ignore_index=True) if bfcl_parts else pd.DataFrame()
bfcl_stat=pd.DataFrame(bfcl_status)
bfcl.to_csv(RESULTS/"10_bfcl_metrics.csv",index=False)
bfcl_stat.to_csv(RESULTS/"10_bfcl_status.csv",index=False)
display(bfcl_stat)

# Stage B — AgentDojo

Uses `openai-compatible`, `--model-id`, fresh log directories, and `--force-rerun`, with fallback within each model family.

In [ ]:
# CELL 8 — AGENTDOJO ENVIRONMENT
DOJO=WORK/"agentdojo"
if SELFTEST:
    DOJO_COMMIT="SELFTEST"
else:
    DOJO_COMMIT,git_status=clone("https://github.com/ethz-spylab/agentdojo.git",DOJO)
    (ENV/"agentdojo_git_status.txt").write_text(git_status)
    ensure_uv()
    p=run(["uv","sync"],cwd=DOJO); save_log("agentdojo_sync.log",p)
    if p.returncode:raise RuntimeError("AgentDojo install failed")
    hp=run(["uv","run","python","-m","agentdojo.scripts.benchmark","--help"],cwd=DOJO); save_log("agentdojo_help.log",hp)
    txt=(hp.stdout or "")+(hp.stderr or "")
    if not all(x in txt for x in ["openai-compatible","--model-id","--force-rerun"]):
        raise RuntimeError("AgentDojo CLI does not expose required interface")
    fr=run(["uv","pip","freeze"],cwd=DOJO); (ENV/"agentdojo_pip_freeze.txt").write_text(fr.stdout or "")
save_json(ENV/"agentdojo_version.json",{"commit":DOJO_COMMIT})
print("AgentDojo:",DOJO_COMMIT)

In [ ]:
# CELL 9 — AGENTDOJO EXECUTION / FALLBACK / PARSING
def parse_dojo(root,label,family,suite):
    rows=[]; root=Path(root)
    for p in root.rglob("*.json"):
        try:o=json.loads(p.read_text())
        except Exception:continue
        if not isinstance(o,dict):continue
        u=o.get("utility"); sec=o.get("security")
        if not isinstance(u,bool) and not isinstance(sec,bool):continue
        rows.append({"benchmark":"AgentDojo","model":label,"family":family,"suite":suite,
                     "user_task_id":o.get("user_task_id"),"injection_task_id":o.get("injection_task_id"),
                     "utility":np.nan if not isinstance(u,bool) else int(u),
                     "security":np.nan if not isinstance(sec,bool) else int(sec),
                     "error":o.get("error"),"source_file":str(p.relative_to(root))})
    return pd.DataFrame(rows)

dojo_parts=[]; dojo_status=[]

if SELFTEST:
    for s in selected:
        suite="banking"; root=RAW/"agentdojo"/s["family"]/suite/"selftest"; root.mkdir(parents=True,exist_ok=True)
        (root/"case.json").write_text(json.dumps({"user_task_id":"user_task_0","injection_task_id":"injection_task_0",
                                                 "utility":True,"security":True,"error":None}))
        d=parse_dojo(root,s["label"],s["family"],suite); d["evidence_state"]="SELFTEST_ONLY"
        dojo_parts.append(d); dojo_status.append({"family":s["family"],"suite":suite,"status":"SELFTEST_ONLY","n":len(d)})
else:
    env=os.environ.copy()
    env["OPENAI_COMPATIBLE_BASE_URL"]=API_BASE
    env["OPENAI_COMPATIBLE_API_KEY"]=OPENROUTER_API_KEY
    for s in selected:
        family=s["family"]; candidate_order=[]
        for m in [s["model"]]+family_candidates.get(family,[]):
            if m not in candidate_order:candidate_order.append(m)
        for suite in CFG["dojo_suites"]:
            key=f"dojo::{family}::{suite}::{MODE}"
            root=RAW/"agentdojo"/family/suite/MODE.lower(); root.mkdir(parents=True,exist_ok=True)
            if supported(key):
                d=parse_dojo(root,family.lower(),family,suite)
                if len(d):
                    dojo_parts.append(d)
                    dojo_status.append({"family":family,"suite":suite,"model_id":ck_get(key).get("model_id"),
                                        "status":"SUPPORTED_REUSED","n":len(d)})
                    continue
            success=False; attempts=[]
            for idx,model_id in enumerate(candidate_order[:5],1):
                cmd=["uv","run","python","-m","agentdojo.scripts.benchmark",
                     "--model","openai-compatible","--model-id",model_id,
                     "-s",suite,"--attack","important_instructions",
                     "--logdir",str(root),"--force-rerun","--max-workers","1"]
                if CFG["dojo_user_tasks"] is not None:
                    for ut in CFG["dojo_user_tasks"]:cmd += ["-ut",ut]
                if CFG["dojo_injection_tasks"] is not None:
                    for it in CFG["dojo_injection_tasks"]:cmd += ["-it",it]
                p=run(cmd,cwd=DOJO,env=env,timeout=None); save_log(f"dojo_{family}_{suite}_{idx}.log",p,cmd)
                d=parse_dojo(root,family.lower(),family,suite)
                valid=int(((d.utility.notna())|(d.security.notna())).sum()) if len(d) else 0
                errs=int(d.error.notna().sum()) if len(d) and "error" in d else 0
                errclass=classify_error((p.stdout or "")+"\n"+(p.stderr or ""))
                attempts.append({"model_id":model_id,"returncode":p.returncode,"valid":valid,"errors":errs,"error_class":errclass})
                if valid>0 and errs==0:
                    d["model_id"]=model_id; dojo_parts.append(d)
                    dojo_status.append({"family":family,"suite":suite,"model_id":model_id,"status":"SUPPORTED","n":valid})
                    ck_set(key,"SUPPORTED",model_id=model_id,n=valid); success=True; break
                if valid>0:
                    d["model_id"]=model_id; dojo_parts.append(d)
                if errclass not in {"PROVIDER_BUDGET","RATE_LIMIT","MODEL_UNAVAILABLE","CONTEXT_LIMIT"}:break
            save_json(RESULTS/f"dojo_{family}_{suite}_attempts.json",attempts)
            if not success:
                st="PARTIAL" if any(a["valid"]>0 for a in attempts) else "INFRA_FAILURE"
                n=max([a["valid"] for a in attempts] or [0])
                dojo_status.append({"family":family,"suite":suite,"model_id":None,"status":st,"n":n})
                ck_set(key,st,attempts=attempts)

dojo=pd.concat(dojo_parts,ignore_index=True) if dojo_parts else pd.DataFrame()
dojo_stat=pd.DataFrame(dojo_status)
dojo.to_csv(RESULTS/"11_agentdojo_cases.csv",index=False)
dojo_stat.to_csv(RESULTS/"11_agentdojo_status.csv",index=False)
display(dojo_stat)

# Stage C — τ³ / tau2-bench

Uses dependency verification inside tau2's own `.venv`, one fixed user simulator across agent models, bounded token/step budgets, `--auto-resume`, model fallback within each family, and strict `results.json` parsing.

In [ ]:
# CELL 10 — TAU ENVIRONMENT
TAU=WORK/"tau2-bench"
if SELFTEST:
    TAU_COMMIT="SELFTEST"
else:
    TAU_COMMIT,tau_git_status=clone("https://github.com/sierra-research/tau2-bench.git",TAU)
    (ENV/"tau_git_status.txt").write_text(tau_git_status)
    ensure_uv()
    p=run(["uv","sync"],cwd=TAU); save_log("tau_sync.log",p)
    if p.returncode:raise RuntimeError("tau2 sync failed")
    TAUPY=TAU/".venv"/"bin"/"python"
    dep=run(["uv","pip","install","--python",TAUPY,"websockets","soundfile"],cwd=TAU); save_log("tau_deps.log",dep)
    ver=run([TAUPY,"-c","import websockets,soundfile; print('TAU_READY')"],cwd=TAU); save_log("tau_verify.log",ver)
    if ver.returncode or "TAU_READY" not in ver.stdout:raise RuntimeError("tau dependency verification failed")
    fr=run(["uv","pip","freeze"],cwd=TAU); (ENV/"tau_pip_freeze.txt").write_text(fr.stdout or "")
save_json(ENV/"tau_version.json",{"commit":TAU_COMMIT})
print("tau2:",TAU_COMMIT)

In [ ]:
# CELL 11 — TAU EXECUTION / FALLBACK / STRICT PARSER
def parse_tau(p):
    p=Path(p)
    try:o=json.loads(p.read_text())
    except Exception:return pd.DataFrame()
    sims=[]
    if isinstance(o,list):sims=o
    elif isinstance(o,dict):
        for k in ["simulations","results","trajectories"]:
            if isinstance(o.get(k),list):sims=o[k];break
    rows=[]
    for i,s in enumerate(sims):
        if not isinstance(s,dict):continue
        reward=None
        if isinstance(s.get("reward_info"),dict) and isinstance(s["reward_info"].get("reward"),(int,float,bool)):
            reward=float(s["reward_info"]["reward"])
        elif isinstance(s.get("reward"),(int,float,bool)):
            reward=float(s["reward"])
        err=s.get("error")
        if err is None and isinstance(s.get("info"),dict):err=s["info"].get("error")
        rows.append({"trajectory_index":i,"task_id":s.get("task_id"),"trial":s.get("trial"),"reward":reward,"error":err})
    return pd.DataFrame(rows)

def model_cost(mid): return price_proxy(byid.get(mid,{}))
FIXED_USER=min([s["model"] for s in selected],key=model_cost) if selected else None

tau_parts=[]; tau_status=[]

if SELFTEST:
    for s in selected:
        for domain in CFG["tau_domains"]:
            root=RAW/"tau"/s["family"]/domain/"selftest"; root.mkdir(parents=True,exist_ok=True)
            (root/"results.json").write_text(json.dumps({"simulations":[
                {"task_id":"0","trial":0,"reward_info":{"reward":1.0},"error":None},
                {"task_id":"1","trial":0,"reward_info":{"reward":0.0},"error":None}]}))
            d=parse_tau(root/"results.json")
            d["benchmark"]="tau3";d["model"]=s["label"];d["family"]=s["family"];d["domain"]=domain;d["evidence_state"]="SELFTEST_ONLY"
            tau_parts.append(d); tau_status.append({"family":s["family"],"domain":domain,"status":"SELFTEST_ONLY","n":int(d.reward.notna().sum())})
else:
    env=os.environ.copy(); env["OPENROUTER_API_KEY"]=OPENROUTER_API_KEY
    for s in selected:
        family=s["family"]; candidate_order=[]
        for m in [s["model"]]+family_candidates.get(family,[]):
            if m not in candidate_order:candidate_order.append(m)
        for domain in CFG["tau_domains"]:
            key=f"tau::{family}::{domain}::{MODE}"
            archive_root=RAW/"tau"/family/domain/MODE.lower(); archive_root.mkdir(parents=True,exist_ok=True)
            if supported(key) and (archive_root/"results.json").exists():
                d=parse_tau(archive_root/"results.json")
                if d.reward.notna().any():
                    d["benchmark"]="tau3";d["model"]=family.lower();d["family"]=family;d["domain"]=domain
                    tau_parts.append(d)
                    tau_status.append({"family":family,"domain":domain,"model_id":ck_get(key).get("model_id"),
                                       "status":"SUPPORTED_REUSED","n":int(d.reward.notna().sum())})
                    continue
            success=False; attempts=[]
            for idx,model_id in enumerate(candidate_order[:5],1):
                run_name=f"ntx_closure_{family.lower()}_{domain}_{MODE.lower()}_{idx}"
                live=TAU/"data"/"simulations"/run_name
                cmd=["uv","run","tau2","run","--domain",domain,
                     "--agent-llm","openrouter/"+model_id,"--user-llm","openrouter/"+FIXED_USER,
                     "--agent-llm-args",json.dumps({"temperature":0.0,"max_tokens":CFG["tau_agent_max_tokens"]}),
                     "--user-llm-args",json.dumps({"temperature":0.0,"max_tokens":CFG["tau_user_max_tokens"]}),
                     "--num-trials","1","--task-split-name","base","--max-steps",str(CFG["tau_max_steps"]),
                     "--max-errors","3","--max-concurrency","1","--max-retries","1","--retry-delay","2",
                     "--seed",str(SEED),"--save-to",run_name,"--auto-resume","--verbose-logs","--llm-log-mode","all"]
                if CFG["tau_tasks_per_domain"] is not None:cmd += ["--num-tasks",str(CFG["tau_tasks_per_domain"])]
                p=run(cmd,cwd=TAU,env=env,timeout=None); save_log(f"tau_{family}_{domain}_{idx}.log",p,cmd)
                if live.exists():copytree(live,archive_root)
                rp=archive_root/"results.json"; d=parse_tau(rp) if rp.exists() else pd.DataFrame()
                n=int(d.reward.notna().sum()) if len(d) else 0
                infra=int(d.error.notna().sum()) if len(d) and "error" in d else 0
                errclass=classify_error((p.stdout or "")+"\n"+(p.stderr or ""))
                attempts.append({"model_id":model_id,"returncode":p.returncode,"n":n,"infra_errors":infra,"error_class":errclass})
                if n>0 and infra==0:
                    d["benchmark"]="tau3";d["model"]=family.lower();d["family"]=family;d["domain"]=domain;d["model_id"]=model_id
                    tau_parts.append(d); tau_status.append({"family":family,"domain":domain,"model_id":model_id,"status":"SUPPORTED","n":n})
                    ck_set(key,"SUPPORTED",model_id=model_id,n=n,user_model=FIXED_USER); success=True; break
                if n>0:
                    d["benchmark"]="tau3";d["model"]=family.lower();d["family"]=family;d["domain"]=domain;d["model_id"]=model_id
                    tau_parts.append(d)
                if errclass not in {"PROVIDER_BUDGET","RATE_LIMIT","MODEL_UNAVAILABLE","CONTEXT_LIMIT"}:break
            save_json(RESULTS/f"tau_{family}_{domain}_attempts.json",attempts)
            if not success:
                n=max([a["n"] for a in attempts] or [0]); st="PARTIAL" if n>0 else "INFRA_FAILURE"
                tau_status.append({"family":family,"domain":domain,"model_id":None,"status":st,"n":n})
                ck_set(key,st,attempts=attempts)

tau=pd.concat(tau_parts,ignore_index=True) if tau_parts else pd.DataFrame()
tau_stat=pd.DataFrame(tau_status)
tau.to_csv(RESULTS/"14_tau_cases.csv",index=False)
tau_stat.to_csv(RESULTS/"14_tau_status.csv",index=False)
display(tau_stat)

In [ ]:
# CELL 12 — OPTIONAL EXTENSION STATUS
optional=[
    {"benchmark":"AgentDyn","status":"DISABLED" if not ENABLE_AGENTDYN else "OPTIONAL_NOT_REQUIRED_FOR_CLOSURE"},
    {"benchmark":"MLCL","status":"OFFICIAL_SOURCE_PRESENT" if MLCL_OFFICIAL_PATH and Path(MLCL_OFFICIAL_PATH).exists() else "MISSING_OFFICIAL_SOURCE"},
    {"benchmark":"MCP-SafetyBench","status":"DISABLED_FOR_SAFETY" if not ENABLE_MCP else "MANUAL_ISOLATED_RUN_REQUIRED"},
]
optional_df=pd.DataFrame(optional)
optional_df.to_csv(RESULTS/"15_optional_status.csv",index=False)
display(optional_df)

In [ ]:
# CELL 13 — UNIFIED NATIVE EVIDENCE
evidence=[]
if len(bfcl):
    for _,r in bfcl.iterrows():
        evidence.append({"benchmark":"BFCL-v4","family":r.get("family"),"model":r.get("model"),
                         "slice":r.get("slice"),"metric":r.get("metric"),"score":r.get("score"),
                         "n":np.nan,"evidence_state":"SELFTEST_ONLY" if SELFTEST else "SUPPORTED"})
if len(dojo):
    for _,r in dojo.iterrows():
        if pd.notna(r.get("utility")):
            evidence.append({"benchmark":"AgentDojo","family":r["family"],"model":r["model"],"slice":r["suite"],
                             "metric":"utility","score":float(r["utility"]),"n":1,
                             "evidence_state":"SELFTEST_ONLY" if SELFTEST else "SUPPORTED"})
        if pd.notna(r.get("security")):
            evidence.append({"benchmark":"AgentDojo","family":r["family"],"model":r["model"],"slice":r["suite"],
                             "metric":"security","score":float(r["security"]),"n":1,
                             "evidence_state":"SELFTEST_ONLY" if SELFTEST else "SUPPORTED"})
if len(tau):
    for _,r in tau.iterrows():
        if pd.notna(r.get("reward")):
            evidence.append({"benchmark":"tau3","family":r["family"],"model":r["model"],"slice":r["domain"],
                             "metric":"reward","score":float(r["reward"]),"n":1,
                             "evidence_state":"SELFTEST_ONLY" if SELFTEST else "SUPPORTED"})

evidence=pd.DataFrame(evidence)
evidence.to_csv(RESULTS/"20_unified_external_evidence.csv",index=False)
if len(evidence):
    summary=(evidence.groupby(["benchmark","family","model","slice","metric","evidence_state"],dropna=False)
             .agg(n_cases=("score","count"),mean_score=("score","mean"),min_score=("score","min"),max_score=("score","max"))
             .reset_index())
else:
    summary=pd.DataFrame(columns=["benchmark","family","model","slice","metric","evidence_state","n_cases","mean_score","min_score","max_score"])
summary.to_csv(RESULTS/"20_external_summary.csv",index=False)
display(summary)

In [ ]:
# CELL 14 — BOOTSTRAP CIs
def boot(values,B):
    x=np.asarray(pd.Series(values).dropna(),float)
    if len(x)==0:return np.nan,np.nan,np.nan
    if len(x)==1:return float(x[0]),np.nan,np.nan
    rng=np.random.default_rng(SEED); means=np.empty(B)
    for i in range(B):means[i]=rng.choice(x,size=len(x),replace=True).mean()
    return float(x.mean()),float(np.quantile(means,.025)),float(np.quantile(means,.975))

ci_rows=[]
if len(evidence):
    for (b,fam,sl,metric),g in evidence.groupby(["benchmark","family","slice","metric"]):
        mean,lo,hi=boot(g.score,CFG["bootstrap"])
        ci_rows.append({"benchmark":b,"family":fam,"slice":sl,"metric":metric,"n":len(g),
                        "mean":mean,"ci95_low":lo,"ci95_high":hi})
ci=pd.DataFrame(ci_rows)
ci.to_csv(RESULTS/"21_bootstrap_ci.csv",index=False)
display(ci)

In [ ]:
# CELL 15 — STRICT PAPER-CLOSURE CLAIM GATE
required_families={s["family"] for s in selected}

bf_ok=(not SELFTEST and len(bfcl_stat)>0 and required_families.issubset(
    set(bfcl_stat.loc[bfcl_stat.status.astype(str).str.startswith("SUPPORTED"),"family"].astype(str))
))

dojo_ok=False
if not SELFTEST and len(dojo_stat):
    good=dojo_stat[dojo_stat.status.astype(str).str.startswith("SUPPORTED")]
    dojo_ok=True
    for fam in required_families:
        have=set(good.loc[good.family==fam,"suite"].astype(str))
        if not set(CFG["dojo_suites"]).issubset(have):
            dojo_ok=False;break

tau_ok=False
if not SELFTEST and len(tau_stat):
    good=tau_stat[tau_stat.status.astype(str).str.startswith("SUPPORTED")]
    tau_ok=True
    for fam in required_families:
        have=set(good.loc[good.family==fam,"domain"].astype(str))
        if not set(CFG["tau_domains"]).issubset(have):
            tau_ok=False;break

claims=[
    {"claim":"BFCL-v4 external function-calling validation","status":"SELFTEST_ONLY" if SELFTEST else ("SUPPORTED" if bf_ok else "MISSING")},
    {"claim":"AgentDojo external prompt-injection validation","status":"SELFTEST_ONLY" if SELFTEST else ("SUPPORTED" if dojo_ok else "MISSING")},
    {"claim":"tau3 stateful external validation","status":"SELFTEST_ONLY" if SELFTEST else ("SUPPORTED" if tau_ok else "PARTIAL" if len(tau) else "MISSING")},
    {"claim":">=3 independent external model families","status":"SELFTEST_ONLY" if SELFTEST else ("SUPPORTED" if len(required_families)>=3 else "MISSING")},
    {"claim":"External-validation closure gate","status":"SELFTEST_ONLY" if SELFTEST else ("SUPPORTED" if bf_ok and dojo_ok and tau_ok and len(required_families)>=3 else "INCOMPLETE")},
]
claims_df=pd.DataFrame(claims)
claims_df.to_csv(RESULTS/"22_paper_closure_claims.csv",index=False)
display(claims_df)

In [ ]:
# CELL 16 — PAPER TABLES + READY-TO-PASTE EXTERNAL SECTION
summary.to_latex(PAPER/"external_benchmark_summary.tex",index=False,float_format="%.4f")
ci.to_latex(PAPER/"external_bootstrap_ci.tex",index=False,float_format="%.4f")
claims_df.to_latex(PAPER/"external_claim_gate.tex",index=False)

compact=[]
if len(evidence):
    for (b,fam,metric),g in evidence.groupby(["benchmark","family","metric"]):
        compact.append({"benchmark":b,"family":fam,"metric":metric,"n":len(g),"mean":float(g.score.mean())})
compact_df=pd.DataFrame(compact)
compact_df.to_csv(PAPER/"external_compact_table.csv",index=False)
compact_df.to_latex(PAPER/"external_compact_table.tex",index=False,float_format="%.4f")

gate=claims_df.loc[claims_df.claim=="External-validation closure gate","status"].iloc[0]

if SELFTEST:
    section=r'''\paragraph{External validation.}
This paragraph was generated in NON-PAPER SELFTEST mode and must not be used in a manuscript.
'''
elif gate=="SUPPORTED":
    fams=", ".join(sorted(required_families))
    section=f'''\\paragraph{{External validation.}}
We evaluated the runtime on three independent external benchmark families:
BFCL-v4 for function calling, AgentDojo for prompt-injection robustness,
and $\\tau^3$ for stateful customer-service tool use. The completed matrix
covers the independent model families {fams}. Table~\\ref{{tab:external-closure}}
reports benchmark-native metrics; unlike metrics are not pooled into a single
score. All reported rows originate from the official benchmark runners and
contain at least one evaluated case.
'''
else:
    available=[]
    for b in ["BFCL-v4","AgentDojo","tau3"]:
        if len(evidence[evidence.benchmark==b]):available.append(b)
    available_text=", ".join(available) if available else "no required external benchmark"
    section=f'''\\paragraph{{External validation.}}
The external-validation matrix remains incomplete. Native evidence was obtained
for {available_text}. We therefore retain the internal frozen evaluation as the
primary effectiveness claim and treat the available external rows as bounded
transfer evidence only. No unsupported cross-benchmark or three-family
generalization claim is made.
'''

(PAPER/"external_validation_section.tex").write_text(section.strip()+"\n")
(PAPER/"README.md").write_text(
    f"# External-validation manuscript integration\n\nClosure gate: **{gate}**\n\n"
    f"Selected model families: {', '.join(sorted(required_families))}\n\n"
    "Use the generated TeX/CSV files directly when updating the manuscript.\n"
)
print((PAPER/"README.md").read_text())

In [ ]:
# CELL 17 — FIGURES
if len(summary):
    for (b,metric),g in summary.groupby(["benchmark","metric"]):
        rows=[]
        for fam,fg in g.groupby("family"):
            w=np.maximum(fg.n_cases.to_numpy(dtype=float),1)
            rows.append({"family":fam,"mean":float(np.average(fg.mean_score.to_numpy(dtype=float),weights=w))})
        agg=pd.DataFrame(rows).sort_values("mean")
        if not len(agg):continue
        fig,ax=plt.subplots(figsize=(8,max(3,0.55*len(agg)+1)))
        ax.barh(agg.family,agg["mean"]);ax.set_title(f"{b}: {metric}");ax.set_xlabel(metric);fig.tight_layout()
        safe=re.sub(r"[^A-Za-z0-9]+","_",f"{b}_{metric}").strip("_")
        fig.savefig(PAPER/f"{safe}.png",dpi=240,bbox_inches="tight")
        plt.show()

In [ ]:
# CELL 18 — PROVENANCE MANIFEST
manifest={
    "experiment":"NTX-PAPER-CLOSURE","created_at":now(),"mode":MODE,"selftest":SELFTEST,
    "selected_models":selected,"fixed_tau_user_model":FIXED_USER if "FIXED_USER" in globals() else None,
    "claims":claims_df.to_dict("records"),"previous_results_import":previous_import,"config":CFG,
}
hashes={}
for root_name,root in [("results",RESULTS),("raw",RAW),("paper",PAPER),("logs",LOGS),("environment",ENV)]:
    for p in sorted(root.rglob("*")):
        if p.is_file() and p.name not in {"FINAL_MANIFEST.json","SHA256SUMS.txt"}:
            hashes[f"{root_name}/{p.relative_to(root)}"]=sha256_file(p)
manifest["artifact_sha256"]=hashes
save_json(RESULTS/"FINAL_MANIFEST.json",manifest)
(RESULTS/"SHA256SUMS.txt").write_text("\n".join(f"{h}  {k}" for k,h in sorted(hashes.items()))+"\n")
print("Manifest files:",len(hashes))

In [ ]:
# CELL 19 — FOUR FINAL ARCHIVES
raw_stage=BASE/"_raw_stage"; shutil.rmtree(raw_stage,ignore_errors=True); raw_stage.mkdir()
copytree(RAW,raw_stage/"raw"); copytree(LOGS,raw_stage/"logs"); copytree(ENV,raw_stage/"environment")

RAW_ZIP=ARCH/"NTX_CLOSURE_RAW_DATA.zip"
PROCESSED_ZIP=ARCH/"NTX_CLOSURE_PROCESSED_RESULTS.zip"
PAPER_ZIP=ARCH/"NTX_CLOSURE_PAPER_INTEGRATION.zip"
MASTER_ZIP=ARCH/"NTX_PAPER_CLOSURE_MASTER.zip"

make_zip(raw_stage,RAW_ZIP); make_zip(RESULTS,PROCESSED_ZIP); make_zip(PAPER,PAPER_ZIP)

master_stage=BASE/"_master_stage"; shutil.rmtree(master_stage,ignore_errors=True); master_stage.mkdir()
for p in [RAW_ZIP,PROCESSED_ZIP,PAPER_ZIP]:shutil.copy2(p,master_stage/p.name)
copytree(RESULTS,master_stage/"results");copytree(PAPER,master_stage/"paper_integration")
copytree(ENV,master_stage/"environment");copytree(LOGS,master_stage/"logs")
make_zip(master_stage,MASTER_ZIP)

archive_rows=[]
for p in [RAW_ZIP,PROCESSED_ZIP,PAPER_ZIP,MASTER_ZIP]:
    archive_rows.append({"file":p.name,"size_mib":round(p.stat().st_size/1024**2,3),"sha256":sha256_file(p)})
archive_df=pd.DataFrame(archive_rows)
archive_df.to_csv(ARCH/"ARCHIVE_MANIFEST.csv",index=False)
display(archive_df)

In [ ]:
# CELL 20 — FINAL STATUS + DOWNLOAD
gate=claims_df.loc[claims_df.claim=="External-validation closure gate","status"].iloc[0]
final={"closure_gate":gate,"mode":MODE,"selftest":SELFTEST,"selected_families":sorted(required_families),
       "claims":claims_df.to_dict("records"),"master_zip":str(MASTER_ZIP),"master_sha256":sha256_file(MASTER_ZIP)}
save_json(ARCH/"FINAL_CLOSURE_STATUS.json",final)
print(json.dumps(final,indent=2))
try:
    from google.colab import files
    for p in [PAPER_ZIP,PROCESSED_ZIP,RAW_ZIP,MASTER_ZIP]:files.download(str(p))
except Exception as e:
    print("Browser auto-download unavailable:",repr(e))
    print("Files remain in:",ARCH)